# Module 7 — Day 5: Evaluation Harness

**Goal:** Build an evaluation suite for the agentic multimodal RAG system. Measure retrieval quality, answer correctness, grounding/citation accuracy, and tool-use efficiency, then compare the agentic workflow with the Module 6 non-agentic baseline.

**Evaluation coverage:** text-only questions, text-plus-image questions, and multi-hop questions.

**Prerequisites:** Module 6 artifacts and Module 7 Days 1–4. Upload the Module 6 `day4_retrieval_results.json`, the Module 7 Day 2 routing results, and the Module 7 Day 4 text-image results.

## 1. Install dependencies
Use pandas and scikit-learn for retrieval metrics and tabular evaluation.

In [1]:
!pip -q install pandas scikit-learn

## 2. Imports and workspace
Create a place to store evaluation cases, metric outputs, and run traces.

In [2]:
import json, re
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

WORK = Path("module7_day5")
WORK.mkdir(exist_ok=True)

## 3. Upload available run artifacts
Upload `day4_retrieval_results.json` from Module 6 and, if available, `day2_query_plan_and_routing.json` and `day4_text_image_agent_results.json` from Module 7. Missing files are allowed.

In [3]:
from google.colab import files
uploaded = files.upload()
json_paths = [Path(x) for x in uploaded if x.lower().endswith(".json")]
def find_file(fragment):
    return next((p for p in json_paths if fragment.lower() in p.name.lower()), None)

m6_path = find_file("day4_retrieval_results")
day2_path = find_file("day2_query_plan_and_routing")
day4_path = find_file("day4_text_image_agent_results")
print("Module 6 retrieval:", m6_path)
print("Day 2 routing:", day2_path)
print("Day 4 image agent:", day4_path)

Saving day4_text_image_agent_results.json to day4_text_image_agent_results.json
Saving day2_query_plan_and_routing.json to day2_query_plan_and_routing.json
Saving day4_retrieval_results.json to day4_retrieval_results.json
Module 6 retrieval: day4_retrieval_results.json
Day 2 routing: day2_query_plan_and_routing.json
Day 4 image agent: day4_text_image_agent_results.json


## 4. Define a small gold evaluation suite
Each case has a query type, expected answer facts, expected source IDs (when known), and a reference response. Replace or extend these examples with facts and source IDs verified from your own invoice and retrieval outputs.

In [4]:
eval_cases = [
    {
        "case_id":"TXT-01","query_type":"text_only",
        "question":"Who is the supplier?",
        "reference_answer":"NORTHSTAR OFFICE SUPPLIES",
        "expected_facts":["NORTHSTAR OFFICE SUPPLIES"],
        "expected_sources":[],
        "notes":"Verify the supplier name against the sample invoice."
    },
    {
        "case_id":"TXT-02","query_type":"text_only",
        "question":"What is the invoice total?",
        "reference_answer":"₹9,086.00",
        "expected_facts":["9086"],
        "expected_sources":[],
        "notes":"The expected amount should be checked against the exact invoice currency and formatting."
    },
    {
        "case_id":"IMG-01","query_type":"text_plus_image",
        "question":"Find a document visual similar to the supplied query image.",
        "reference_answer":"A relevant extracted document image is retrieved and identified by filename.",
        "expected_facts":[],
        "expected_sources":[],
        "notes":"Human reviewer must confirm visual relevance; do not score image filename overlap as visual correctness."
    },
    {
        "case_id":"MH-01","query_type":"multi_hop",
        "question":"Compare the chart on page 2 with the totals table on page 1.",
        "reference_answer":"The answer should cite the chart evidence from page 2 and the totals table evidence from page 1, then compare only values visible in those sources.",
        "expected_facts":[],
        "expected_sources":[],
        "notes":"Requires both sources and human verification of the comparison."
    }
]
eval_df = pd.DataFrame(eval_cases)
display(eval_df[["case_id","query_type","question","reference_answer"]])

,case_id,query_type,question,reference_answer
0,TXT-01,text_only,Who is the supplier?,NORTHSTAR OFFICE SUPPLIES
1,TXT-02,text_only,What is the invoice total?,"₹9,086.00"
2,IMG-01,text_plus_image,Find a document visual similar to the supplied...,A relevant extracted document image is retriev...
3,MH-01,multi_hop,Compare the chart on page 2 with the totals ta...,The answer should cite the chart evidence from...


## 5. Load Module 6 retrieval records when available
These records provide the non-agentic baseline corpus and known source identifiers.

In [17]:
m6_records = []

if m6_path:
    raw = json.loads(m6_path.read_text(encoding="utf-8"))

    result_groups = [
        ("text_only", raw.get("text_only_results", [])),
        ("image_only", raw.get("image_only_results", [])),
        ("text_plus_image", raw.get("text_plus_image_results", []))
    ]

    seen = set()

    for group_name, items in result_groups:
        for item in items:
            if not isinstance(item, dict):
                continue

            record_id = item.get("record_id")
            if record_id in seen:
                continue
            seen.add(record_id)

            content = item.get("text") or item.get("content") or ""
            if not content:
                continue

            m6_records.append({
                "source_id": record_id or f"M6-{len(m6_records)+1}",
                "source_doc": item.get("source_doc", "Module 6"),
                "page": item.get("page"),
                "text": str(content),
                "modality": item.get("modality", "text"),
                "result_group": group_name,
                "similarity": item.get("similarity")
            })

print("Module 6 records:", len(m6_records))
display(pd.DataFrame(m6_records))

Module 6 records: 16


,source_id,source_doc,page,text,modality,result_group,similarity
0,text_7,sample_multimodal_invoice.pdf,1.0,Items and Services,text,text_only,0.9032
1,text_2,sample_multimodal_invoice.pdf,1.0,Supplier,text,text_only,0.9009
2,text_11,sample_multimodal_invoice.pdf,1.0,Grand Total,text,text_only,0.8890
3,text_21,sample_multimodal_invoice.pdf,2.0,Service Summary,text,text_only,0.8839
4,text_13,sample_multimodal_invoice.pdf,1.0,Notes: Goods supplied are covered by the manuf...,text,text_only,0.8820
5,text_1,sample_multimodal_invoice.pdf,1.0,Original for recipient,text,text_only,0.8762
6,text_14,sample_multimodal_invoice.pdf,1.0,Authorised Signatory,text,text_only,0.8729
7,text_17,sample_multimodal_invoice.pdf,2.0,Service and Delivery Details,text,text_only,0.8707
8,image_1,sample_multimodal_invoice.pdf,NaN,the table shows the different types of product...,image,image_only,0.7591
9,image_4,sample_multimodal_invoice.pdf,NaN,table showing the different types of services,image,image_only,0.7154


## 6. Implement the Module 6-style non-agentic baseline
The baseline performs one retrieval pass for each text query. It does not plan, route, reformulate, or re-retrieve.

In [18]:
def baseline_retrieve(query, records, top_k=5):
    if not records:
        return []
    texts=[r["text"] for r in records]
    vec=TfidfVectorizer(stop_words="english",ngram_range=(1,2))
    mat=vec.fit_transform(texts+[query])
    scores=cosine_similarity(mat[-1],mat[:-1]).ravel()
    order=scores.argsort()[::-1][:min(top_k,len(records))]
    return [{**records[i],"score":round(float(scores[i]),4)} for i in order]

baseline_demo = baseline_retrieve("invoice total supplier",m6_records)
print(json.dumps(baseline_demo[:3],indent=2)[:3500])

[
  {
    "source_id": "text_2",
    "source_doc": "sample_multimodal_invoice.pdf",
    "page": 1,
    "text": "Supplier",
    "modality": "text",
    "result_group": "text_only",
    "similarity": 0.9009,
    "score": 0.4174
  },
  {
    "source_id": "text_11",
    "source_doc": "sample_multimodal_invoice.pdf",
    "page": 1,
    "text": "Grand Total",
    "modality": "text",
    "result_group": "text_only",
    "similarity": 0.889,
    "score": 0.1814
  },
  {
    "source_id": "image_2",
    "source_doc": "sample_multimodal_invoice.pdf",
    "page": null,
    "text": "the table shows the total number of employees and the number of employees",
    "modality": "image",
    "result_group": "image_only",
    "similarity": 0.5999,
    "score": 0.0704
  }
]


## 7. Load optional agentic run artifacts
Extract source IDs and tool-call counts where the saved artifacts contain them. These are execution observations, not correctness labels.

In [19]:
day2_data = json.loads(day2_path.read_text(encoding="utf-8")) if day2_path else {}
day4_data = json.loads(day4_path.read_text(encoding="utf-8")) if day4_path else {}

def count_tool_calls(data):
    trace = data.get("routing_trace", data.get("agent_trace", [])) if isinstance(data,dict) else []
    return len(trace), [x.get("tool") for x in trace if isinstance(x,dict)]

print("Day 2 trace:",count_tool_calls(day2_data)[0],"calls")
print("Day 4 trace:",count_tool_calls(day4_data)[0],"calls")

Day 2 trace: 8 calls
Day 4 trace: 4 calls


## 8. Define evaluation metrics
We measure retrieval precision@k and recall@k when expected source IDs are known, answer fact coverage for text answers, citation validity, and tool-use efficiency. Human review fields capture correctness and visual grounding that automatic lexical checks cannot reliably judge.

In [20]:
def precision_at_k(retrieved_ids, relevant_ids, k=5):
    if not relevant_ids: return np.nan
    top=set(retrieved_ids[:k])
    return len(top & set(relevant_ids))/min(k,len(retrieved_ids)) if retrieved_ids else 0.0

def recall_at_k(retrieved_ids, relevant_ids, k=5):
    if not relevant_ids: return np.nan
    return len(set(retrieved_ids[:k]) & set(relevant_ids))/len(set(relevant_ids))

def fact_coverage(answer, expected_facts):
    if not expected_facts: return np.nan
    normalized=re.sub(r"[^a-z0-9.]","",str(answer).lower())
    hits=sum(1 for fact in expected_facts if re.sub(r"[^a-z0-9.]","",fact.lower()) in normalized)
    return hits/len(expected_facts)

def citation_validity(cited_ids, known_ids):
    if not cited_ids: return np.nan
    if not known_ids: return np.nan
    return sum(1 for x in cited_ids if x in known_ids)/len(cited_ids)

print("Metric functions ready.")

Metric functions ready.


## 9. Create a run-results table for manual scoring
Enter the actual agent answer, retrieved source IDs, citations, and tool calls for each test case. The template is intentionally editable because the notebook cannot infer semantic correctness from filenames or similarity scores.

In [21]:
run_rows = []
for case in eval_cases:
    run_rows.append({
        "case_id":case["case_id"],
        "system":"agentic",
        "answer":"",
        "retrieved_source_ids":"",
        "cited_source_ids":"",
        "tool_calls":0,
        "human_answer_correct":None,
        "human_grounding_correct":None,
        "visual_relevance_correct":None,
        "review_notes":""
    })
agent_results_df = pd.DataFrame(run_rows)
display(agent_results_df)

,case_id,system,answer,retrieved_source_ids,cited_source_ids,tool_calls,human_answer_correct,human_grounding_correct,visual_relevance_correct,review_notes
0,TXT-01,agentic,,,,0,None,None,None,
1,TXT-02,agentic,,,,0,None,None,None,
2,IMG-01,agentic,,,,0,None,None,None,
3,MH-01,agentic,,,,0,None,None,None,


## 10. Score answer coverage and citations
After filling the table above, run this cell to calculate automatic checks. Leave unknown human judgments blank rather than guessing.

In [22]:
known_source_ids={r["source_id"] for r in m6_records}
case_lookup={c["case_id"]:c for c in eval_cases}
scored=[]
for _,row in agent_results_df.iterrows():
    case=case_lookup[row["case_id"]]
    answer=str(row["answer"])
    retrieved=[x.strip() for x in str(row["retrieved_source_ids"]).split(",") if x.strip()]
    cited=[x.strip() for x in str(row["cited_source_ids"]).split(",") if x.strip()]
    scored.append({
        "case_id":row["case_id"],
        "fact_coverage":fact_coverage(answer,case["expected_facts"]),
        "citation_validity":citation_validity(cited,known_source_ids),
        "tool_calls":row["tool_calls"],
        "human_answer_correct":row["human_answer_correct"],
        "human_grounding_correct":row["human_grounding_correct"],
        "visual_relevance_correct":row["visual_relevance_correct"]
    })
agent_scores=pd.DataFrame(scored)
display(agent_scores)

,case_id,fact_coverage,citation_validity,tool_calls,human_answer_correct,human_grounding_correct,visual_relevance_correct
0,TXT-01,0.0,NaN,0,None,None,None
1,TXT-02,0.0,NaN,0,None,None,None
2,IMG-01,NaN,NaN,0,None,None,None
3,MH-01,NaN,NaN,0,None,None,None


## 11. Compare baseline and agentic retrieval
For text-only and multi-hop cases, populate expected source IDs from your verified gold labels. This cell computes precision@5 and recall@5 when labels are present.

In [23]:
retrieval_comparison=[]
for case in eval_cases:
    relevant=case["expected_sources"]
    if case["query_type"]=="text_plus_image":
        continue
    hits=baseline_retrieve(case["question"],m6_records,top_k=5)
    ids=[h["source_id"] for h in hits]
    retrieval_comparison.append({
        "case_id":case["case_id"],"system":"module6_baseline",
        "precision@5":precision_at_k(ids,relevant,5),
        "recall@5":recall_at_k(ids,relevant,5),
        "retrieved_ids":", ".join(ids)
    })
display(pd.DataFrame(retrieval_comparison))

,case_id,system,precision@5,recall@5,retrieved_ids
0,TXT-01,module6_baseline,NaN,NaN,"text_2, text_4, image_3, text_26, image_5"
1,TXT-02,module6_baseline,NaN,NaN,"text_11, image_2, text_26, text_4, image_5"
2,MH-01,module6_baseline,NaN,NaN,"image_4, image_3, image_1, image_2, text_4"


## 12. Evaluate tool-use efficiency
Compare the number of calls used by each system. The baseline normally uses one retrieval call; the agentic count should come from the recorded tool trace. Fewer calls are not automatically better if answer quality falls.

In [24]:
efficiency_rows=[
    {"system":"module6_baseline","tool_calls":1,"note":"Single retrieval pass per query."},
    {"system":"module7_agentic","tool_calls":count_tool_calls(day2_data)[0]+count_tool_calls(day4_data)[0],
     "note":"Count from uploaded Day 2 and Day 4 traces; add Day 3 calls if available."}
]
efficiency_df=pd.DataFrame(efficiency_rows)
display(efficiency_df)

,system,tool_calls,note
0,module6_baseline,1,Single retrieval pass per query.
1,module7_agentic,12,Count from uploaded Day 2 and Day 4 traces; ad...


## 13. Human evaluation rubric
Use this rubric for every case. For image and multi-hop questions, inspect the original image/document and verify that the cited sources actually support the answer.

In [25]:
rubric = pd.DataFrame([
    {"dimension":"Answer correctness","score_0":"Incorrect or unsupported","score_1":"Partially correct","score_2":"Correct and complete"},
    {"dimension":"Grounding/citations","score_0":"No valid supporting source","score_1":"Some claims supported; citation gaps","score_2":"All key claims supported by cited sources"},
    {"dimension":"Visual relevance","score_0":"Wrong or irrelevant image","score_1":"Partially relevant image","score_2":"Correct visual evidence retrieved"},
    {"dimension":"Multi-hop completeness","score_0":"Misses a required hop","score_1":"Finds some required sources","score_2":"Finds and connects all required sources"},
    {"dimension":"Tool-use efficiency","score_0":"Unnecessary/repeated calls","score_1":"Some avoidable calls","score_2":"Only useful calls; stops appropriately"}
])
display(rubric)

,dimension,score_0,score_1,score_2
0,Answer correctness,Incorrect or unsupported,Partially correct,Correct and complete
1,Grounding/citations,No valid supporting source,Some claims supported; citation gaps,All key claims supported by cited sources
2,Visual relevance,Wrong or irrelevant image,Partially relevant image,Correct visual evidence retrieved
3,Multi-hop completeness,Misses a required hop,Finds some required sources,Finds and connects all required sources
4,Tool-use efficiency,Unnecessary/repeated calls,Some avoidable calls,Only useful calls; stops appropriately


## 14. Summarize the evaluation and success gate
A majority of cases should be correct across text-only, text-plus-image, and multi-hop categories, with grounded citations. At least one case should demonstrate recovery after first-pass retrieval fails (from Day 3). Do not mark the gate passed while required human scores are blank.

In [26]:
def success_gate(scores_df, recovery_demonstrated=False):
    human_cols=["human_answer_correct","human_grounding_correct"]
    available=scores_df[human_cols].notna().all(axis=1)
    if not available.any():
        return {"status":"NOT EVALUATED","reason":"Enter human answer and grounding judgments first."}
    passed=(scores_df.loc[available,human_cols] == 2).all(axis=1)
    majority=passed.mean() > 0.5
    return {
        "status":"PASS" if majority and recovery_demonstrated else "NOT YET PASSED",
        "majority_cases_correct":bool(majority),
        "recovery_demonstrated":bool(recovery_demonstrated),
        "scored_cases":int(available.sum()),
        "reason":"Requires majority correct and at least one documented self-correction case."
    }

print(success_gate(agent_scores,recovery_demonstrated=False))

{'status': 'NOT EVALUATED', 'reason': 'Enter human answer and grounding judgments first.'}


## 15. Save evaluation artifacts
Save the gold suite, editable run table, automatic scores, retrieval comparison, and human rubric for the final report.

In [27]:
def records_to_json(df):
    return df.replace({np.nan:None}).to_dict(orient="records")

artifacts={
    "evaluation_cases":eval_cases,
    "agent_run_input_template":records_to_json(agent_results_df),
    "agent_scores":records_to_json(agent_scores),
    "retrieval_comparison":records_to_json(pd.DataFrame(retrieval_comparison)),
    "efficiency":records_to_json(efficiency_df),
    "human_rubric":records_to_json(rubric)
}
out=WORK/"day5_evaluation_results.json"
out.write_text(json.dumps(artifacts,indent=2,ensure_ascii=False),encoding="utf-8")
agent_results_df.to_csv(WORK/"day5_agent_run_scoring_template.csv",index=False)
print("Saved:",out)
print("Saved:",WORK/"day5_agent_run_scoring_template.csv")

Saved: module7_day5/day5_evaluation_results.json
Saved: module7_day5/day5_agent_run_scoring_template.csv
